# Llama 3.2 QLoRA: No Token Required
**Prepared, not trained.** Uses the public `unsloth/Llama-3.2-1B-Instruct` community distribution of Meta Llama 3.2, not Qwen or Llama-3-8B. Its Meta Llama license still applies.

No Hugging Face token, account, Secrets setup, or Unsloth installation is needed. Choose a free hosted T4 GPU. Downloads explicitly use `token=False`; no weights or GPU packages install on your Mac.

The previously inspected 120-case synthetic benchmark is a retrospective comparison, **not a fresh blind test**. The app and verified Qwen results remain unchanged.


In [ ]:
import sys
from pathlib import Path
try:
    import google.colab
except ImportError:
    raise RuntimeError('Hosted Google Colab only. Do not run this on your Mac.')
if not Path('/content').is_dir():
    raise RuntimeError('Use a hosted runtime, not a local runtime.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Stop and select an available free GPU; do not buy compute.')
if torch.cuda.get_device_properties(0).total_memory < 10*1024**3:
    raise RuntimeError('This experiment needs at least 10 GiB GPU memory.')
print('GPU:', torch.cuda.get_device_name(0))


## Upload the small experiment bundle
Upload **llama_no_token_bundle.zip**. It contains code, synthetic data and cached reference predictions, not weights or credentials.


In [ ]:
from google.colab import files
import io
import stat
import zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload only llama_no_token_bundle.zip')
name, content = next(iter(uploaded.items()))
if not name.endswith('.zip'):
    raise ValueError('Expected a ZIP bundle')
root = Path('/content/llama_no_token_experiment')
if root.exists():
    raise RuntimeError('Folder exists. Use a fresh runtime to avoid mixing runs.')
with zipfile.ZipFile(io.BytesIO(content)) as archive:
    if sum(i.file_size for i in archive.infolist()) > 10*1024**2:
        raise ValueError('Unexpected archive size; this bundle should contain no weights')
    for member in archive.infolist():
        target = (root/member.filename).resolve()
        if root.resolve() not in target.parents or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError('Unsafe archive entry')
    archive.extractall(root)
sys.path.insert(0, str(root))
sys.modules.pop('protocol', None)
from protocol import verify_bundle
manifest = verify_bundle(root)
print(manifest['provenance'])


## Install only in hosted Colab
Retain Colab's existing CUDA PyTorch. This does not install anything on your Mac.


In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(root/'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, str(root/'llama_experiment.py'), 'check'], check=True)


## Check public model access: no token
The public checkpoint is checked anonymously before weight downloads. If the repository becomes gated/private or is unavailable, stop; there is no login workaround or paid fallback. The Meta Llama license still applies.


In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download
repo_id = 'unsloth/Llama-3.2-1B-Instruct'
revision = '5a8abab4a5d6f164389b1079fb721cfab8d7126c'
try:
    info = HfApi().model_info(repo_id, revision=revision, token=False)
    if info.private or info.gated:
        raise RuntimeError('This checkpoint is no longer anonymously available')
    config_path = hf_hub_download(repo_id=repo_id, filename='config.json', revision=revision, token=False)
    if json.loads(Path(config_path).read_text()).get('model_type') != 'llama':
        raise RuntimeError('The public checkpoint is not a standard Llama model')
except Exception:
    raise RuntimeError('Anonymous model check failed. Stop here; share the error, not any credentials.') from None
print('Public Llama checkpoint verified. No token was read or used.')


## Train and compare
Rank-16 NF4 QLoRA, up to 300 steps, effective batch 8, domain-only loss. Classify six support domains. Compare untuned and trained Llama now; previously verified Qwen/TF-IDF/MiniLM/SetFit predictions are **cached references**, not newly executed models. Scores may improve or regress; no automatic deployment occurs.


In [ ]:
results = Path('/content/llama_no_token_results')
subprocess.run([sys.executable, str(root/'llama_experiment.py'), 'run', '--out', str(results)], check=True)
print((results/'RESULTS.md').read_text())


## Download completed results
Includes metrics, training evidence and the safetensors Llama adapter, not optimizer pickle files. Do not publicly redistribute the adapter until Meta license, naming and attribution requirements are satisfied.


In [ ]:
output_zip = Path('/content/llama_qlora_results.zip')
subprocess.run([sys.executable, str(root/'export_results.py'), str(results), str(output_zip)], check=True)
files.download(str(output_zip))


Built with Llama. [Public model and license](https://huggingface.co/unsloth/Llama-3.2-1B-Instruct) | [PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization) | [Colab limits](https://research.google.com/colaboratory/faq.html)

No Llama training or GPU result is established until all training/evaluation cells complete.
